In [ ]:
!pip install -U langchain langchain-openai langchain-core langchain-community

In [ ]:
# ENV SETUP
import os

os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGSMITH_KEY"
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [ ]:
# IMPORTS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

In [ ]:
# INITIALIZE MODEL
llm = ChatOpenAI(model="gpt-3.5-turbo")

extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract:
- Skills
- Experience
- Tools

Resume:
{resume}
"""
)

score_prompt = PromptTemplate(
    input_variables=["resume_data", "job_description"],
    template="""
Compare resume with job description.

Give:
- Score (0-100)
- Explanation

Job Description:
{job_description}

Resume:
{resume_data}
"""
)

# 🔥 IMPORTANT LINE (YOU MISSED THIS)
extract_chain = extract_prompt | llm
score_chain = score_prompt | llm

In [ ]:
# JOB DESCRIPTION
job_description = """
Looking for Data Scientist with:
- Python
- Machine Learning
- Deep Learning
- SQL
- NLP
- 2+ years experience
"""

In [ ]:
# SAMPLE RESUMES
strong_resume = """
Python, Machine Learning, Deep Learning, NLP, SQL
3 years experience as Data Scientist
Worked on real ML projects
"""

average_resume = """
Python, SQL
1 year experience
Basic ML knowledge
"""

weak_resume = """
Excel, Communication
No programming
Fresher
"""

In [ ]:
# SKILL EXTRACTION PROMPT
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract:
- Skills
- Experience
- Tools

Resume:
{resume}

Do NOT assume anything.
"""
)

In [ ]:
# MATCH + SCORE PROMPT
score_prompt = PromptTemplate(
    input_variables=["resume_data", "job_description"],
    template="""
Compare resume with job description.

Return:
Score (0-100)
Explanation

Job Description:
{job_description}

Resume:
{resume_data}

Rules:
- Be strict
- Do NOT assume missing skills
"""
)

In [ ]:
# CHAINS
def extract_info(resume):
    prompt = f"Extract skills, tools, experience:\n{resume}"
    return free_llm(prompt)
score_chain = score_prompt | llm

In [ ]:
!pip install transformers

from transformers import pipeline

# Free model
generator = pipeline("text-generation", model="gpt2")

job_description = """
Looking for Data Scientist with:
Python, Machine Learning, NLP, SQL
"""
# PIPELINE FUNCTION
def evaluate_resume(resume):

    print("\n========================")
    print("PROCESSING RESUME")
    print("========================")

    # Step 1: Extraction
    extracted = generator(
        f"Extract skills, tools, experience:\n{resume}",
        max_length=120
    )[0]["generated_text"]

    print("\n--- Extracted Info ---")
    print(extracted)

    # Step 2: Scoring
    result = generator(
        f"""
        Compare resume with job description:

        {job_description}

        Resume:
        {extracted}

        Give score (0-100) and explanation.
        """,
        max_length=200
    )[0]["generated_text"]

    print("\n--- Final Result ---")
    print(result)

In [ ]:
# RUN ALL 3 CASES
evaluate_resume(strong_resume)
evaluate_resume(average_resume)
evaluate_resume(weak_resume)

In [ ]:
# DEBUG CASE
print("\n🔹 DEBUG TEST (Missing Data)")

bad_resume = "I know nothing about tech"

evaluate_resume(bad_resume)